# Preprocessing: data analysis and visualization

This notebook goes with the optional preprocessing of `framework/preprocessing/` and the
corresponding sections of `train_config.yaml`. It reads the **same config** as the training,
so what is shown here is what the training will do.

1. **Band order check**: required for NDVI / NDWI
2. **Tile statistics**: why tiles help with class imbalance
3. **Re-sampling simulation**: distribution before / after, variability between trainings, over-fitting risk
4. **Class weights** resulting from the distribution
5. **Data augmentation**: each transform separately, then the full random pipeline
6. **Spectral indices**: NDVI / NDWI maps and per-class distributions
7. **Pipeline preview**: real batches as fed to the model

Run it from the repository root (or from a sub-folder: the root is detected).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'framework').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import yaml
import tensorflow as tf

from framework.dataset import LandCoverData as LCD, numpy_parse_image
from framework.utils import YamlNamespace
from framework.preprocessing.config import cfg_get
from framework.preprocessing import tile_stats as ts
from framework.preprocessing import sampling
from framework.preprocessing import spectral as spectral_mod
from framework.preprocessing import pipeline
from framework.preprocessing.augmentation import (
    Augmenter, random_flip_rot90, random_zoom,
    random_brightness, random_contrast, random_color, gaussian_noise)
from framework.preprocessing.class_weights import compute_class_weights

CONFIG_FILE = ROOT / 'train_config.yaml'
with open(CONFIG_FILE) as f:
    CONFIG = YamlNamespace(yaml.safe_load(f))


def resolve(path):
    # config paths are relative to the repository root
    p = Path(path).expanduser()
    return p if p.is_absolute() else (ROOT / p).resolve()


DATASET = resolve(CONFIG.dataset_folder)
SAMPLE_IDS = sorted(int(p.stem) for p in (DATASET / 'train' / 'images').glob('*.tif'))
CLASSES = list(LCD.CLASSES)
VALID = [i for i in range(LCD.N_CLASSES) if i not in LCD.IGNORED_CLASSES_IDX]
CMAP = ListedColormap(np.array([LCD.CLASSES_COLORPALETTE[c] for c in range(LCD.N_CLASSES)]) / 255)
LEGEND = [Patch(color=CMAP(c), label=CLASSES[c]) for c in range(LCD.N_CLASSES)]
print(f"{len(SAMPLE_IDS)} training images in {DATASET}")

In [ ]:
BAND_ORDER = dict(spectral_mod.DEFAULT_BAND_ORDER)
_bo = cfg_get(CONFIG, 'spectral_indices.band_order', None)
if _bo is not None:
    BAND_ORDER.update(vars(_bo) if not isinstance(_bo, dict) else _bo)


def load_sample(sample_id):
    # uint16 image (256, 256, 4) and uint8 mask (256, 256)
    path = str(DATASET / 'train' / 'images' / f'{sample_id}.tif').encode()
    image, mask = numpy_parse_image(path)
    return image, mask[..., 0]


def stretch_params(image, low=1, high=99):
    # per-band percentiles of the R, G, B bands, to display several versions with the SAME stretch
    x = np.asarray(image, dtype=np.float64)[..., [BAND_ORDER['red'], BAND_ORDER['green'], BAND_ORDER['blue']]]
    return np.percentile(x, low, axis=(0, 1)), np.percentile(x, high, axis=(0, 1))


def to_rgb(image, params=None):
    x = np.asarray(image, dtype=np.float64)[..., [BAND_ORDER['red'], BAND_ORDER['green'], BAND_ORDER['blue']]]
    lo, hi = params if params is not None else stretch_params(image)
    return np.clip((x - lo) / (hi - lo + 1e-12), 0, 1)


def show_mask(mask, ax, title=None):
    ax.imshow(np.squeeze(mask), cmap=CMAP, vmin=0, vmax=LCD.N_CLASSES - 1, interpolation='nearest')
    ax.set_axis_off()
    if title:
        ax.set_title(title)


def show_rgb(image, ax, title=None, params=None):
    ax.imshow(to_rgb(image, params))
    ax.set_axis_off()
    if title:
        ax.set_title(title)

## 1. Band order check

The images are documented as R, G, B, NIR, and NDVI / NDWI rely on it. Physical expectations:
* **NIR** (near infrared) is **much higher than red** on vegetation (broadleaf, coniferous, herbaceous, cultivated);
* on **water**, NIR is **the lowest band** (water absorbs infrared).

If the table contradicts this, fix `spectral_indices.band_order` in `train_config.yaml`.

In [ ]:
rng = np.random.default_rng(0)
subset_ids = rng.choice(SAMPLE_IDS, size=min(300, len(SAMPLE_IDS)), replace=False)

sums = np.zeros((LCD.N_CLASSES, LCD.N_CHANNELS))
counts = np.zeros(LCD.N_CLASSES)
for sid in subset_ids:
    image, mask = load_sample(sid)
    for c in np.unique(mask):
        sel = mask == c
        sums[c] += image[sel].astype(np.float64).sum(0)
        counts[c] += sel.sum()

band_names = [f'band{i}' for i in range(LCD.N_CHANNELS)]
band_means = pd.DataFrame(sums / np.maximum(counts, 1)[:, None], index=CLASSES, columns=band_names)
band_means['pixels'] = counts.astype(int)
band_means = band_means[band_means['pixels'] > 0]
display(band_means.round(1))

ax = band_means[band_names].plot.bar(figsize=(12, 4), rot=0)
ax.set_ylabel('mean raw pixel value')
ax.set_title(f'Mean value of each band per class ({len(subset_ids)} random images)')
plt.show()

nir, red = f"band{BAND_ORDER['nir']}", f"band{BAND_ORDER['red']}"
veg = [c for c in ['broadleaf', 'coniferous', 'herbaceous', 'cultivated'] if c in band_means.index]
print(f"NIR / red ratio on vegetation (expected > 1.5): "
      f"{(band_means.loc[veg, nir] / band_means.loc[veg, red]).round(2).to_dict()}")
if 'water' in band_means.index:
    print(f"Lowest band on water (expected: {nir}): {band_means.loc['water', band_names].idxmin()}")

## 2. Tile statistics

Per-tile class pixel counts (`framework/preprocessing/tile_stats.py`), computed once and cached
in `tiling.tile_stats_csv` (it takes a minute or two the first time).

Key point: **splitting into tiles does not change the global class proportions**: the same pixels
are there. What changes is that tiles are **purer**: a rare class covers a larger share of the
tile it appears in, and fewer tiles contain it. This is what makes it possible to over-sample
rare classes selectively (section 3), without over-sampling the dominant ones at the same time.

In [ ]:
TILE_SIZE = int(cfg_get(CONFIG, 'tiling.tile_size', 128))
STATS_CSV = resolve(cfg_get(CONFIG, 'tiling.tile_stats_csv', DATASET.parent / f'tile_stats_{TILE_SIZE}.csv'))
tile_stats = ts.load_or_compute_tile_stats(STATS_CSV, DATASET, TILE_SIZE)
img_counts = ts.image_class_counts(tile_stats)
print(f"{len(tile_stats)} tiles of {TILE_SIZE}px from {len(img_counts)} images")
tile_stats.head()

In [ ]:
total = img_counts.sum()
dist = pd.DataFrame({'pixels': total, 'share (valid classes)': total / total.iloc[VALID].sum()})
dist.loc[[CLASSES[i] for i in LCD.IGNORED_CLASSES_IDX], 'share (valid classes)'] = np.nan
display(dist.style.format({'pixels': '{:,.0f}', 'share (valid classes)': '{:.4%}'}))

ax = dist['share (valid classes)'].iloc[VALID].plot.bar(figsize=(10, 3.5), logy=True, rot=0,
                                                        color=[CMAP(c) for c in VALID], edgecolor='k')
ax.set_title('Global pixel share of each class (log scale)')
plt.show()

In [ ]:
MIN_FRAC = float(cfg_get(CONFIG, 'tiling.resampling.min_pixel_fraction', 0.01))


def presence_table(counts_df, level):
    c = counts_df[CLASSES].to_numpy(np.float64)
    share = c / c.sum(1, keepdims=True).clip(min=1)
    present = share >= MIN_FRAC
    return pd.DataFrame({
        f'% {level}s containing it': present.mean(0) * 100,
        f'mean share when present ({level})': np.where(present.any(0), (share * present).sum(0) / present.sum(0).clip(min=1), np.nan) * 100,
    }, index=CLASSES)


purity = pd.concat([presence_table(img_counts, 'image'), presence_table(tile_stats, 'tile')], axis=1).iloc[VALID]
display(purity.round(2))
print(f"(a class is 'present' when it covers >= {MIN_FRAC:.0%} of the image / tile)")

fig, axs = plt.subplots(1, 2, figsize=(14, 3.5))
purity[['% images containing it', '% tiles containing it']].plot.bar(ax=axs[0], rot=0)
axs[0].set_title('How often a class is present')
purity[['mean share when present (image)', 'mean share when present (tile)']].plot.bar(ax=axs[1], rot=0)
axs[1].set_title('Share of the image / tile covered when present (%)')
plt.show()

In [ ]:
def n_classes(counts_df):
    c = counts_df[CLASSES].to_numpy(np.float64)[:, VALID]
    return ((c / c.sum(1, keepdims=True).clip(min=1)) >= MIN_FRAC).sum(1)


ignored_cols = [CLASSES[i] for i in LCD.IGNORED_CLASSES_IDX]
fig, axs = plt.subplots(1, 2, figsize=(14, 3.5))
bins = np.arange(0.5, len(VALID) + 1.5)
axs[0].hist([n_classes(img_counts), n_classes(tile_stats)], bins=bins, density=True,
            label=['images 256px', f'tiles {TILE_SIZE}px'])
axs[0].set_xlabel('number of classes present'); axs[0].legend(); axs[0].set_title('Tiles are purer')
ign = tile_stats[ignored_cols].sum(1) / tile_stats[CLASSES].sum(1)
axs[1].hist(ign, bins=50, log=True)
axs[1].set_xlabel('share of ignored pixels (no_data + clouds)'); axs[1].set_title('Clouds per tile')
plt.show()
for thr in (0.25, 0.5, 0.9):
    print(f"tiles with > {thr:.0%} ignored pixels: {(ign > thr).sum()} ({(ign > thr).mean():.2%})")

In [ ]:
# rare classes: how many distinct images / tiles really carry them (limit of any re-sampling)
for cls in ['snow', 'natural', 'water', 'artificial']:
    n_img = int((img_counts[cls] > 0).sum())
    n_tile = int((tile_stats[cls] / tile_stats[CLASSES].sum(1) >= MIN_FRAC).sum())
    print(f"{cls:<11s} pixels in {n_img:5d} images, >= {MIN_FRAC:.0%} of the tile in {n_tile:5d} tiles")

## 3. Re-sampling simulation

`sampling.build_manifest` with the parameters of `train_config.yaml` (section `tiling`). A random 10 %
of the images plays the validation set here (in training: `val_samples_csv` or the random split of `train.py`).

* `share_ratio` = effective share / natural share: > 1 means the class is seen more often than in the raw data.
* Each seed is a different training: the intact part and the stochastic repetitions change.

In [ ]:
val_csv = cfg_get(CONFIG, 'val_samples_csv', None)
if val_csv:
    val_ids = set(pd.read_csv(resolve(val_csv))['sample_id'].astype(int))
else:
    val_ids = set(np.random.default_rng(42).choice(SAMPLE_IDS, int(0.1 * len(SAMPLE_IDS)), replace=False).tolist())
train_ids = [i for i in SAMPLE_IDS if i not in val_ids]

RS = dict(
    resampling=bool(cfg_get(CONFIG, 'tiling.resampling.enabled', True)),
    min_pixel_fraction=MIN_FRAC,
    threshold=float(cfg_get(CONFIG, 'tiling.resampling.threshold', 0.1)),
    max_repeat=float(cfg_get(CONFIG, 'tiling.resampling.max_repeat', 4)),
    max_ignored_fraction=float(cfg_get(CONFIG, 'tiling.max_ignored_fraction', 1.0)),
)
INTACT_FRACTION = float(cfg_get(CONFIG, 'tiling.intact_fraction', 0.3))
print(f"intact_fraction={INTACT_FRACTION}, {RS}")

manifest, report, summary, effective = sampling.build_manifest(
    train_ids, tile_stats, INTACT_FRACTION, np.random.default_rng(0), **RS)
print(summary)
display(report.style.format({
    'natural_pixels': '{:,.0f}', 'effective_pixels': '{:,.0f}', 'natural_share': '{:.4%}',
    'effective_share': '{:.4%}', 'share_ratio': '{:.2f}', 'pool_tile_frequency': '{:.4f}',
    'class_repeat_factor': '{:.2f}'}))

In [ ]:
r = report.iloc[VALID]
x = np.arange(len(r))
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
axs[0].bar(x - 0.2, r['natural_share'], 0.4, label='natural')
axs[0].bar(x + 0.2, r['effective_share'], 0.4, label='after re-sampling')
axs[0].set_yscale('log'); axs[0].set_xticks(x); axs[0].set_xticklabels(r.index); axs[0].legend()
axs[0].set_title('Pixel share of each class (log)')
axs[1].bar(x, r['share_ratio'], color=[CMAP(c) for c in VALID], edgecolor='k')
axs[1].axhline(1, color='k', lw=1); axs[1].set_xticks(x); axs[1].set_xticklabels(r.index)
axs[1].set_title('effective / natural share')
plt.show()

In [ ]:
# variability between trainings: 5 draws (= 5 trainings with resample_seed: null)
draws = {}
for seed in range(5):
    m, rep, summ, _ = sampling.build_manifest(train_ids, tile_stats, INTACT_FRACTION,
                                              np.random.default_rng(seed), **RS)
    draws[seed] = (m, rep, summ)

intact_sets = {s: set(m.loc[m.tile_idx == -1, 'sample_id']) for s, (m, _, _) in draws.items()}
overlap = pd.DataFrame([[len(intact_sets[a] & intact_sets[b]) / max(1, len(intact_sets[a])) for b in draws]
                        for a in draws], index=draws, columns=draws)
print('Share of intact images in common between two trainings '
      f'(expected ~ intact_fraction = {INTACT_FRACTION}):')
display(overlap.round(3))
display(pd.DataFrame({s: rep['effective_share'] for s, (_, rep, _) in draws.items()}).iloc[VALID]
        .style.format('{:.4%}').set_caption('effective share per draw'))
display(pd.DataFrame({s: summ for s, (_, _, summ) in draws.items()}).T)

In [ ]:
# over-fitting risk: how many times the same tile comes back, per class
tiles_m = manifest[manifest.tile_idx != -1]
reps = tiles_m.groupby(['sample_id', 'tile_idx']).size().rename('copies').reset_index()
reps = reps.merge(tile_stats, on=['sample_id', 'tile_idx'])
share = reps[CLASSES].to_numpy(np.float64) / reps[CLASSES].sum(1).to_numpy()[:, None]
rows = []
for c in VALID:
    sel = share[:, c] >= MIN_FRAC
    rows.append({'class': CLASSES[c], 'unique tiles': int(sel.sum()),
                 'copies in manifest': int(reps.loc[sel, 'copies'].sum()),
                 'mean copies per tile': reps.loc[sel, 'copies'].mean() if sel.any() else np.nan})
display(pd.DataFrame(rows).set_index('class').round(2))

plt.figure(figsize=(6, 3))
reps['copies'].value_counts().sort_index().plot.bar(rot=0)
plt.title('Number of copies of a tile in the manifest'); plt.show()

In [ ]:
# sensitivity to the RFS threshold (other parameters from the config)
rows = []
for thr in [0.0, 0.02, 0.05, 0.1, 0.2, 0.3]:
    p = dict(RS, threshold=thr)
    _, rep, summ, _ = sampling.build_manifest(train_ids, tile_stats, INTACT_FRACTION, np.random.default_rng(0), **p)
    row = {'threshold': thr, 'manifest tiles': summ['n_manifest_tiles']}
    row.update({c: rep.loc[c, 'share_ratio'] for c in ['artificial', 'natural', 'water', 'snow', 'cultivated']})
    rows.append(row)
display(pd.DataFrame(rows).set_index('threshold').round(2).style.set_caption('share_ratio per class'))

## 4. Class weights of the loss

`class_weights.compute_class_weights`: `w(c) = freq(c)^-power`, normalized to an average pixel weight
of 1 and clipped to `max_weight`. With re-sampling, the weights are computed on the **effective**
distribution (`counts_source: manifest`), otherwise rare classes would be compensated twice.

In [ ]:
MAX_W = float(cfg_get(CONFIG, 'class_weighting.max_weight', 10.0))
cw = {}
for p in (0.0, 0.25, 0.5, 1.0):
    cw[f'static p={p}'] = compute_class_weights(LCD.TRAIN_CLASS_COUNTS, power=p, max_weight=MAX_W)
    cw[f'manifest p={p}'] = compute_class_weights(effective, power=p, max_weight=MAX_W)
display(pd.DataFrame(cw, index=CLASSES).round(3))

## 5. Data augmentation

Transforms of `framework/preprocessing/augmentation.py` (TensorFlow native). First each transform
alone, at the **bounds of its range** in the config, then the full random pipeline.
All versions are displayed with the **same contrast stretch** as the original, so brightness /
contrast / color changes are visible as they are (a per-image stretch would hide them).

In [ ]:
AUG_ENABLED = bool(cfg_get(CONFIG, 'augmentation.enabled', False))
AUG = Augmenter.from_config(CONFIG) or Augmenter()   # config values, or defaults if disabled
print(f"augmentation.enabled = {AUG_ENABLED} -> parameters displayed: {AUG.params}")

# a sample with several classes
n_cls = n_classes(img_counts)
demo_id = int(img_counts.index[np.argmax(n_cls)])
image_u16, mask_np = load_sample(demo_id)
image = tf.cast(image_u16, tf.float32) / LCD.TRAIN_PIXELS_MAX      # same as dataset.normalize
mask = tf.constant(mask_np[..., None])
params = stretch_params(image.numpy())
print(f"demo sample: {demo_id}")

In [ ]:
P = AUG.params
tf.random.set_seed(1)
z_img, z_msk = random_zoom(image, mask, P['zoom']['min_scale'], P['zoom']['min_scale'])
f_img, f_msk = random_flip_rot90(image, mask)
b_lo, b_hi = P['brightness']['range']
c_lo, c_hi = P['contrast']['range']
g_lo, g_hi = P['color']['range']
versions = [
    ('original', image, mask),
    ('flip + rot90', f_img, f_msk),
    (f"zoom x{1 / P['zoom']['min_scale']:.2f}", z_img, z_msk),
    (f'brightness x{b_lo}', random_brightness(image, b_lo, b_lo), mask),
    (f'brightness x{b_hi}', random_brightness(image, b_hi, b_hi), mask),
    (f'contrast x{c_lo}', random_contrast(image, c_lo, c_lo), mask),
    (f'contrast x{c_hi}', random_contrast(image, c_hi, c_hi), mask),
    (f'color gains in [{g_lo}, {g_hi}]', random_color(image, g_lo, g_hi), mask),
    (f"noise std={P['noise']['std']}", gaussian_noise(image, P['noise']['std']), mask),
]
fig, axs = plt.subplots(2, len(versions), figsize=(3 * len(versions), 6.5))
for j, (title, img, msk) in enumerate(versions):
    show_rgb(img.numpy(), axs[0, j], title, params)
    show_mask(msk.numpy(), axs[1, j])
fig.legend(handles=LEGEND, loc='lower center', ncol=LCD.N_CLASSES)
plt.suptitle('Each transform alone (top: image, bottom: mask)'); plt.show()

In [ ]:
# full random pipeline: 8 draws (geometric then photometric), mask alignment check
n_draws = 8
fig, axs = plt.subplots(2, n_draws + 1, figsize=(3 * (n_draws + 1), 6.5))
show_rgb(image.numpy(), axs[0, 0], 'original', params); show_mask(mask.numpy(), axs[1, 0])
for j in range(1, n_draws + 1):
    img, msk = AUG(image, mask)
    assert set(np.unique(msk.numpy())) <= set(np.unique(mask_np)), 'zoom created a new label value'
    show_rgb(img.numpy(), axs[0, j], f'draw {j}', params); show_mask(msk.numpy(), axs[1, j])
fig.legend(handles=LEGEND, loc='lower center', ncol=LCD.N_CLASSES)
plt.suptitle('Full augmentation pipeline'); plt.show()

In [ ]:
# overlay: mask contours on the augmented image, to check the geometric alignment
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax in axs:
    img, msk = AUG.geometric(image, mask)
    ax.imshow(to_rgb(img.numpy(), params))
    ax.contour(np.squeeze(msk.numpy()), levels=np.arange(LCD.N_CLASSES) + 0.5, colors='yellow', linewidths=0.6)
    ax.set_axis_off()
plt.suptitle('Mask contours over the geometrically augmented image'); plt.show()

In [ ]:
# effect on the band statistics over 500 draws (photometric only)
stats = []
for _ in range(500):
    x = AUG.photometric(image).numpy()
    stats.append(x.mean((0, 1)) / image.numpy().mean((0, 1)))
stats = pd.DataFrame(stats, columns=band_names)
stats.plot.hist(bins=40, alpha=0.5, figsize=(8, 3))
plt.title('Band mean after / before photometric augmentation'); plt.show()
display(stats.describe().round(3))

## 6. Spectral indices: NDVI and NDWI

* NDVI = (NIR − R) / (NIR + R): vegetation
* NDWI = (G − NIR) / (G + NIR): open water

The per-class distributions show how much information each index carries for each class.

In [ ]:
SPECTRAL = spectral_mod.SpectralIndices(['ndvi', 'ndwi'], BAND_ORDER)
idx = SPECTRAL.compute(image).numpy()
fig, axs = plt.subplots(1, 5, figsize=(22, 4.5))
show_rgb(image.numpy(), axs[0], 'RGB', params)
axs[1].imshow(image.numpy()[..., BAND_ORDER['nir']], cmap='gray'); axs[1].set_title('NIR'); axs[1].set_axis_off()
for ax, k, name, cmap in [(axs[2], 0, 'NDVI', 'RdYlGn'), (axs[3], 1, 'NDWI', 'RdBu')]:
    im = ax.imshow(idx[..., k], cmap=cmap, vmin=-1, vmax=1); ax.set_title(name); ax.set_axis_off()
    plt.colorbar(im, ax=ax, fraction=0.046)
show_mask(mask_np, axs[4], 'mask')
plt.show()

In [ ]:
# per-class distributions on the random subset (pixels sub-sampled)
rng = np.random.default_rng(0)
records = []
for sid in subset_ids[:150]:
    img_u16, m = load_sample(sid)
    ind = SPECTRAL.compute(tf.cast(img_u16, tf.float32)).numpy().reshape(-1, 2)
    m = m.ravel()
    pick = rng.choice(len(m), 2000, replace=False)
    records.append(pd.DataFrame({'class': np.array(CLASSES)[m[pick]], 'NDVI': ind[pick, 0], 'NDWI': ind[pick, 1]}))
px = pd.concat(records)
order = [CLASSES[c] for c in VALID if CLASSES[c] in set(px['class'])]
fig, axs = plt.subplots(1, 2, figsize=(16, 4))
for ax, col in zip(axs, ['NDVI', 'NDWI']):
    ax.boxplot([px.loc[px['class'] == c, col] for c in order], labels=order, showfliers=False)
    ax.set_title(f'{col} per class'); ax.axhline(0, color='k', lw=0.5)
plt.show()
display(px.groupby('class')[['NDVI', 'NDWI']].median().loc[order].round(3))

## 7. Pipeline preview

Real batches built with the functions of `framework/preprocessing/pipeline.py` from the manifest of
section 3, with the augmentation and spectral indices of the config (forced on here, for display).

In [ ]:
train_map = pipeline.make_train_map_fn(AUG, SPECTRAL)
tiles_m = manifest[manifest.tile_idx != -1].sample(64, random_state=0)
intact_m = manifest[manifest.tile_idx == -1].sample(8, random_state=0)
paths_t = [str(DATASET / 'train' / 'images' / f'{i}.tif') for i in tiles_m.sample_id]
paths_i = [str(DATASET / 'train' / 'images' / f'{i}.tif') for i in intact_m.sample_id]

ds_tiles = tf.data.Dataset.from_tensor_slices((paths_t, tiles_m.tile_idx.to_numpy(np.int32)))\
    .map(pipeline.make_tile_parse_fn(TILE_SIZE)).map(train_map).batch(16)
ds_intact = tf.data.Dataset.from_tensor_slices(paths_i)\
    .map(pipeline.parse_image).map(train_map).batch(4)

tb_img, tb_msk = next(iter(ds_tiles))
ib_img, ib_msk = next(iter(ds_intact))
for name, (bi, bm) in {'tile batch': (tb_img, tb_msk), 'intact batch': (ib_img, ib_msk)}.items():
    print(f"{name:13s} image {tuple(bi.shape)} {bi.dtype.name}, mask {tuple(bm.shape)} {bm.dtype.name}")
ch_names = band_names + SPECTRAL.names
display(pd.DataFrame({'min': tf.reduce_min(tb_img, [0, 1, 2]).numpy(), 'mean': tf.reduce_mean(tb_img, [0, 1, 2]).numpy(),
                      'max': tf.reduce_max(tb_img, [0, 1, 2]).numpy()}, index=ch_names).round(4)
        .style.set_caption('channel statistics of a tile batch (model input)'))

In [ ]:
fig, axs = plt.subplots(3, 8, figsize=(24, 9))
for j in range(8):
    x = tb_img[j].numpy()
    show_rgb(x[..., :LCD.N_CHANNELS], axs[0, j], f'tile {tiles_m.sample_id.iloc[j]}/{tiles_m.tile_idx.iloc[j]}')
    axs[1, j].imshow(x[..., LCD.N_CHANNELS], cmap='RdYlGn', vmin=-1, vmax=1); axs[1, j].set_axis_off()
    show_mask(tb_msk[j].numpy(), axs[2, j])
axs[1, 0].set_title('NDVI', loc='left')
plt.suptitle(f'Augmented {TILE_SIZE}px tiles as fed to the model'); plt.show()